# `locityper_03_budget` — pilot, cost, keep list

Run **in a VWB JupyterLab app** after a one-sample smoke in
[`locityper_02_run_batch.ipynb`](locityper_02_run_batch.ipynb).

This notebook does **not** replace `_02`. `_02` stays the one-sample smoke.
Here we:

1. Draw a random **pilot** (20 samples) from the v9 CRAM manifest
2. Submit **minicram then genotype** on native Google Batch (flags off until you flip them)
3. Estimate **p90 / p95** cost from VM wall time + Nearline `data_transfer_stats`
4. Write `keep.csv` under the run ledger so
   [`locityper_04_dispatch.ipynb`](locityper_04_dispatch.ipynb) can run the rest

Shared ledger: `scripts/vwb_batch/`. Outputs:
`gs://…/batchRuns/locityper/{sample_id}/`. Run metadata:
`gs://…/batchRuns/locityper/runs/{RUN_ID}/`.

Default BED / `vcf_db` are the **toy CYP2D6** smoke catalog from prep. Swap
`BED` / `DB_TAR` to Isaac’s files before any analysis cohort. `SUBMIT_*` stay
**off**. Status: [`batch_monitor.ipynb`](batch_monitor.ipynb).


In [ ]:
from __future__ import annotations

import os
import random
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "scripts" / "vwb_batch" / "dispatch.py").is_file():
            return p
        nested = p / "aou-lr-phase-2" / "scripts" / "vwb_batch" / "dispatch.py"
        if nested.is_file():
            return nested.parents[2]
    raise FileNotFoundError("Cannot find scripts/vwb_batch. Clone kvg/aou-lr-phase-2 and re-run.")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "scripts"))

from vwb_batch import alloc, cohort, cost, dispatch, gcs, plots, poll, registry, submit  # noqa: E402
from vwb_batch.pipelines import locityper as lt  # noqa: E402

SCRATCH = Path.cwd() / "locityper_rw_scratch"
SCRATCH.mkdir(parents=True, exist_ok=True)

OUTPUT_BUCKET_GS = os.environ.get("LT_OUTPUT_BUCKET_GS", "gs://aou-lr-phase2-resources/")
REGION = os.environ.get("DSUB_REGION", "us-central1")
REF_FA = os.environ.get("LOCITYPER_REF_FA", "gs://aou-lr-phase2-resources/locityper/refs/Homo_sapiens_assembly38.fasta")
REF_FAI = os.environ.get("LOCITYPER_REF_FAI", "gs://aou-lr-phase2-resources/locityper/refs/Homo_sapiens_assembly38.fasta.fai")
COUNTS_JF = os.environ.get("LOCITYPER_COUNTS_JF", "gs://aou-lr-phase2-resources/locityper/refs/counts.Homo_sapiens_assembly38.k25.jf")
BED = os.environ.get("LOCITYPER_BED", "gs://aou-lr-phase2-resources/locityper/smoke.bed")
DB_TAR = os.environ.get("LOCITYPER_DB_TAR", "gs://aou-lr-phase2-resources/locityper/vcf_db.smoke.tar.gz")
PRINT_READS_DOCKER = os.environ.get(
    "LOCITYPER_PRINT_READS_DOCKER",
    "us-central1-docker.pkg.dev/broad-dsp-lrma/aou-lr/aou-locityper-print-reads:0.1.2",
)
LOCITYPER_DOCKER = os.environ.get("LOCITYPER_DOCKER", "us-central1-docker.pkg.dev/broad-dsp-lrma/aou-lr/aou-locityper:1.4.5.0")
OUT_PREFIX = os.environ.get("LT_BATCH_OUT_PREFIX", f"{OUTPUT_BUCKET_GS.rstrip('/')}/batchRuns/locityper")
RUN_ID = os.environ.get("LT_RUN_ID") or f"lt-{datetime.now(timezone.utc).strftime('%y%m%d')}-pilot"
COHORT_CSV = os.environ.get("LT_COHORT_CSV", "")
PILOT_N = int(os.environ.get("LT_PILOT_N", "20"))
PILOT_SEED = int(os.environ.get("LT_PILOT_SEED", "1"))
SHARD_SIZE = int(os.environ.get("LT_SHARD_SIZE", "100"))
PARALLELISM = int(os.environ.get("LT_PILOT_PARALLELISM", "20"))
MAX_RUN_DURATION = os.environ.get("LT_MAX_RUN_DURATION", "28800s")
DEFAULT_SEX = "female"

SUBMIT_PILOT_MINICRAM = False
SUBMIT_PILOT_GENOTYPE = False
WRITE_KEEP = True
WRITE_COMPUTE = True
BUDGET_USD = float(os.environ.get("LT_BUDGET_USD", "0") or 0)
RETRY_MARGIN = float(os.environ.get("LT_RETRY_MARGIN", "1.15"))
COST_QUANTILE = float(os.environ.get("LT_COST_QUANTILE", "0.90"))

PATHS = registry.run_paths(output_bucket=OUTPUT_BUCKET_GS.rstrip("/"), pipeline="locityper", run_id=RUN_ID)
print("REPO_ROOT", REPO_ROOT)
print("RUN_ID", RUN_ID)
print("BED", BED)
print("DB_TAR", DB_TAR)
print("BUDGET_USD", BUDGET_USD, "PILOT_N", PILOT_N)
print("SUBMIT_PILOT_MINICRAM", SUBMIT_PILOT_MINICRAM, "SUBMIT_PILOT_GENOTYPE", SUBMIT_PILOT_GENOTYPE)
print("LOCITYPER_DOCKER", LOCITYPER_DOCKER)


## PET + cohort

v9 CRAM manifest unless `LT_COHORT_CSV` is set. Annotates every row with BED / jellyfish / vcf_db.


In [ ]:
project, sa = alloc.require_vwb_env()
if COHORT_CSV:
    all_records = cohort.read_csv(Path(COHORT_CSV))
else:
    manifest = cohort.find_v9_manifest()
    print("MANIFEST:", manifest or "(not found)")
    if manifest is None:
        raise SystemExit("no cohort: set LT_COHORT_CSV or mount the v9 CRAM manifest")
    all_records = cohort.records_from_manifest(
        manifest, ref_fa=REF_FA, ref_fai=REF_FAI, catalog="", sex=DEFAULT_SEX
    )
for rec in all_records:
    rec["ref_fa"] = rec.get("ref_fa") or REF_FA
    rec["ref_fai"] = rec.get("ref_fai") or REF_FAI
    rec["counts_jf"] = rec.get("counts_jf") or COUNTS_JF
    rec["bed"] = rec.get("bed") or BED
    rec["db_tar"] = rec.get("db_tar") or rec.get("locityper_db_tar_gz") or DB_TAR
    rec["sex"] = rec.get("sex") or DEFAULT_SEX
print("cohort:", len(all_records))
pilot_records = cohort.sample_pilot(all_records, PILOT_N, PILOT_SEED)
print("pilot:", len(pilot_records), [r["sample_id"] for r in pilot_records[:8]])
local_cohort = SCRATCH / f"{RUN_ID}.cohort.csv"
local_pilot = SCRATCH / f"{RUN_ID}.pilot.csv"
cohort.write_csv(local_cohort, all_records, fields=cohort.LOCITYPER_FIELDS)
cohort.write_csv(local_pilot, pilot_records, fields=cohort.LOCITYPER_FIELDS)
gcs.upload_file(local_cohort, PATHS.cohort_csv)
gcs.upload_file(local_pilot, PATHS.pilot_csv)
meta = registry.load_run_json(PATHS)
meta.update({"cohort_n": len(all_records), "pilot_n": len(pilot_records), "bed": BED, "db_tar": DB_TAR})
registry.write_run_json(PATHS, meta)
registry.append_event(PATHS, "run_created", cohort_n=len(all_records), pilot_n=len(pilot_records))
print("ledger", PATHS.prefix)


## Submit pilot minicram

Set `SUBMIT_PILOT_MINICRAM = True`.


In [ ]:
project, sa = alloc.require_vwb_env()
todo = list(pilot_records)
print("minicram todo", len(todo))
if not SUBMIT_PILOT_MINICRAM:
    print("SUBMIT_PILOT_MINICRAM is off")
else:
    n_existing = sum(1 for ev in registry.load_events(PATHS) if ev.get("event") == "job_submitted" and ev.get("stage") == "minicram")
    for shard, chunk in cohort.shards(todo, SHARD_SIZE):
        rows = lt.env_rows(chunk, stage="minicram", out_prefix=OUT_PREFIX, project=project)
        prepared = dispatch.prepare_shard(
            paths=PATHS, stage="minicram", shard=n_existing + shard, records=chunk, env_rows=rows,
            build_job=lt.build_job, image=PRINT_READS_DOCKER, out_prefix=OUT_PREFIX,
            project=project, region=REGION, sa=sa, parallelism=PARALLELISM,
            max_run_duration=MAX_RUN_DURATION, scratch=SCRATCH,
        )
        dispatch.maybe_submit(prepared=prepared, paths=PATHS, stage="minicram", shard=n_existing + shard,
                              project=project, region=REGION, do_submit=True)


## Poll ledger jobs

Safe to re-run. Does not submit.


In [ ]:
project, sa = alloc.require_vwb_env()
listing = set(gcs.ls_recursive(OUT_PREFIX.rstrip("/") + "/"))
now = registry.utcnow()
status_rows = []
seen = set()
for ev in registry.load_events(PATHS):
    if ev.get("event") != "job_submitted":
        continue
    job_id = ev["job_id"]
    if job_id in seen:
        continue
    seen.add(job_id)
    try:
        polled = poll.poll_job(job_id=job_id, project=project, region=REGION)
    except RuntimeError as exc:
        print("skip", job_id, exc)
        continue
    registry.append_event(PATHS, "job_polled", stage=ev["stage"], job_id=job_id, state=polled["state"])
    print(job_id, ev["stage"], polled["state"])
    status_rows.extend(poll.reconcile_shard(
        sample_ids=ev.get("sample_ids") or [], stage=ev["stage"], job_id=job_id,
        shard=int(ev.get("shard") or 0), poll=polled,
        expected=lambda sid, st: lt.expected_objects(sid, st, OUT_PREFIX),
        now=now, listing=listing,
    ))
registry.write_status(PATHS, status_rows)
print("status rows", len(status_rows))


## Submit pilot genotype

Set `SUBMIT_PILOT_GENOTYPE = True` after minicram objects exist.


In [ ]:
project, sa = alloc.require_vwb_env()
listing = set(gcs.ls_recursive(OUT_PREFIX.rstrip("/") + "/"))
pending, done = poll.pending_records(pilot_records, stage="minicram", out_prefix=OUT_PREFIX, expected=lt.expected_objects, listing=listing)
gt_pending, gt_done = poll.pending_records(done, stage="genotype", out_prefix=OUT_PREFIX, expected=lt.expected_objects, listing=listing)
print("minicram done/pending", len(done), len(pending), "genotype done/pending", len(gt_done), len(gt_pending))
if not SUBMIT_PILOT_GENOTYPE:
    print("SUBMIT_PILOT_GENOTYPE is off")
elif not gt_pending:
    print("nothing to submit")
else:
    n_existing = sum(1 for ev in registry.load_events(PATHS) if ev.get("event") == "job_submitted" and ev.get("stage") == "genotype")
    for shard, chunk in cohort.shards(gt_pending, SHARD_SIZE):
        rows = lt.env_rows(chunk, stage="genotype", out_prefix=OUT_PREFIX, project=project)
        prepared = dispatch.prepare_shard(
            paths=PATHS, stage="genotype", shard=n_existing + shard, records=chunk, env_rows=rows,
            build_job=lt.build_job, image=LOCITYPER_DOCKER, out_prefix=OUT_PREFIX,
            project=project, region=REGION, sa=sa, parallelism=PARALLELISM,
            max_run_duration=MAX_RUN_DURATION, scratch=SCRATCH,
        )
        dispatch.maybe_submit(prepared=prepared, paths=PATHS, stage="genotype", shard=n_existing + shard,
                              project=project, region=REGION, do_submit=True)


## Cost + keep list

Planning unit is p90 × `RETRY_MARGIN`. Set `BUDGET_USD` to size `keep.csv`.


In [ ]:
listing = set(gcs.ls_recursive(OUT_PREFIX.rstrip("/") + "/"))
status = {(r.get("sample_id"), r.get("stage")): r for r in registry.load_status(PATHS)}
costs = []
for rec in pilot_records:
    sid = rec["sample_id"]
    mini_ok = all(u in listing for u in lt.expected_objects(sid, "minicram", OUT_PREFIX))
    gt_ok = all(u in listing for u in lt.expected_objects(sid, "genotype", OUT_PREFIX))
    if not mini_ok:
        continue
    try:
        network = cost.parse_transfer_stats(gcs.cat(lt.transfer_stats_uri(sid, OUT_PREFIX)))
    except Exception as exc:
        print("skip stats", sid, exc)
        continue
    row = cost.sample_cost_usd(
        minicram_seconds=float(status.get((sid, "minicram"), {}).get("seconds") or 0) or 0.0,
        genotype_seconds=float(status.get((sid, "genotype"), {}).get("seconds") or 0) or 0.0,
        network_bytes=network,
    )
    row["sample_id"] = sid
    row["genotype_done"] = gt_ok
    costs.append(row)
    print(f"{sid} mini={row['minicram_seconds']:.0f}s gt={row['genotype_seconds']:.0f}s GiB={row['network_bytes']/(1024**3):.3f} usd={row['total_usd']:.4f} gt_ok={gt_ok}")
full = [c for c in costs if c["genotype_done"] and c["genotype_seconds"] > 0]
planning_src = full if len(full) >= max(5, PILOT_N // 10) else costs
if not planning_src:
    raise SystemExit("no completed minicram costs yet")
summary = cost.summarize_pilot(planning_src, q=COST_QUANTILE)
unit = summary["planning_usd"] * RETRY_MARGIN
print("n_costed:", summary["n"], "mean:", round(summary["mean_usd"], 4), "p90:", round(summary["p90_usd"], 4))
print("unit with margin:", round(unit, 4))
done_ids = {c["sample_id"] for c in full} or {c["sample_id"] for c in costs}
done_keep = [r for r in pilot_records if r["sample_id"] in done_ids]
n_keep = cost.n_keep(budget_usd=BUDGET_USD, planning_usd=summary["planning_usd"], retry_margin=RETRY_MARGIN, n_available=len(all_records)) if BUDGET_USD > 0 else len(done_keep)
n_keep = max(n_keep, len(done_keep))
pool = [r for r in all_records if r["sample_id"] not in {x["sample_id"] for x in done_keep}]
random.Random(PILOT_SEED).shuffle(pool)
keep = done_keep + pool[: max(0, n_keep - len(done_keep))]
print("BUDGET_USD:", BUDGET_USD, "n_keep:", len(keep), "includes completed pilot:", len(done_keep))
print("est. spend at planning unit:", round(len(keep) * unit, 2))
if WRITE_KEEP and (BUDGET_USD > 0 or done_keep):
    local_keep = SCRATCH / f"{RUN_ID}.keep.csv"
    cohort.write_csv(local_keep, keep, fields=cohort.LOCITYPER_FIELDS)
    gcs.upload_file(local_keep, PATHS.keep_csv)
    registry.append_event(PATHS, "keep_written", n_keep=len(keep), budget_usd=BUDGET_USD, planning_usd=summary["planning_usd"])
    print("keep.csv:", PATHS.keep_csv)
print("Next: locityper_04_dispatch with LT_RUN_ID=%s" % RUN_ID)


## Size production VMs from pilot RSS/CPU

Writes `compute` into `run.json` for `_04`. Caps at Locityper Batch defaults (4 vCPU / 16 GiB minicram, 2 vCPU / 14 GiB genotype), never the WDL 32-vCPU genotype.


In [ ]:
mod = lt._load_submit_batch()
listing = set(gcs.ls_recursive(OUT_PREFIX.rstrip("/") + "/"))
compute_out = {}
for stage in ("minicram", "genotype"):
    rows = []
    for rec in pilot_records:
        uri = lt.resource_stats_uri(rec["sample_id"], stage, OUT_PREFIX)
        if uri not in listing and not gcs.exists(uri):
            continue
        try:
            rows.append(cost.parse_resource_stats(gcs.cat(uri)))
        except Exception as exc:
            print("skip", uri, exc)
    print(f"{stage} resource files:", len(rows))
    if len(rows) < 5:
        print(f"WARN: <5 {stage} samples; leaving default compute")
        continue
    rec = cost.recommend_compute(rows, stage=stage, q=COST_QUANTILE, defaults=mod.DEFAULT_COMPUTE[stage])
    compute_out[stage] = {
        "cpuMilli": rec["cpuMilli"], "memoryMib": rec["memoryMib"], "bootDiskMib": rec["bootDiskMib"],
        "machine": rec["machine"], "p90_rss_mib": round(rec["p90_rss_mib"], 1),
        "p90_cpu_cores": round(rec["p90_cpu_cores"], 2), "n": rec["n"],
    }
    print(stage, compute_out[stage], "default was", rec["default"])
if WRITE_COMPUTE and compute_out:
    meta = registry.load_run_json(PATHS)
    meta["compute"] = compute_out
    registry.write_run_json(PATHS, meta)
    registry.append_event(PATHS, "compute_tuned", compute=compute_out)
    print("wrote compute to", PATHS.run_json)


## Pilot figures


In [ ]:
FIG_DIR = SCRATCH / f"{RUN_ID}.figures"
resources = {"minicram": [], "genotype": []}
listing = set(gcs.ls_recursive(OUT_PREFIX.rstrip("/") + "/"))
for stage in resources:
    for rec in pilot_records:
        uri = lt.resource_stats_uri(rec["sample_id"], stage, OUT_PREFIX)
        if uri not in listing:
            continue
        try:
            resources[stage].append(cost.parse_resource_stats(gcs.cat(uri)))
        except Exception:
            pass
meta = registry.load_run_json(PATHS)
paths = plots.plot_pilot(
    costs=costs if "costs" in dir() else [],
    resources=resources,
    compute=meta.get("compute"),
    out_dir=FIG_DIR,
    budget_usd=BUDGET_USD,
    n_keep=len(keep) if "keep" in dir() else 0,
    planning_usd=(cost.summarize_pilot(full, q=COST_QUANTILE)["planning_usd"] if "full" in dir() and full else 0),
)
for p in paths:
    print(p)
